# Gradient Boosting — Regresor (XGBoost & LightGBM)

**Repositorio:** `autodidact/both/gradient_boosting/`

---

## 1. Idea General e Intuición

El Gradient Boosting Regresor sigue la misma filosofía iterativa que el clasificador, pero la función de pérdida que se minimiza es el **MSE** (por defecto):

$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^n (y_i - F(\mathbf{x}_i))^2$$

Los pseudo-residuos en regresión MSE son simplemente los **residuos ordinarios**:

$$r_{im} = y_i - F_{m-1}(\mathbf{x}_i)$$

Cada árbol aprende a predecir cuánto se equivocó el modelo anterior.

## 2. Fundamento Matemático

### 2.1 Funciones de Pérdida para Regresión

| Pérdida | `objective` (XGB) | `objective` (LGBM) | Uso |
|---|---|---|---|
| **MSE** | `reg:squarederror` | `regression` | Default, sensible a outliers |
| **MAE** | `reg:absoluteerror` | `regression_l1` | Robusto a outliers |
| **Huber** | `reg:pseudohubererror` | `huber` | Intermedio |
| **Poisson** | `count:poisson` | `poisson` | Datos de conteo |
| **Tweedie** | `reg:tweedie` | `tweedie` | Seguros, skewed |

### 2.2 Métricas de Evaluación

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$ | Mismas unidades que $y$ |
| **MAE** | $\frac{1}{n}\sum\|y_i-\hat{y}_i\|$ | Robusto a outliers |
| **$R^2$** | $1-\frac{SS_{res}}{SS_{tot}}$ | Proporción de varianza explicada |
| **MAPE** | $\frac{100}{n}\sum\frac{\|y_i-\hat{y}_i\|}{|y_i|}$ | Error relativo (%) |

### 2.3 SHAP Values (Explicabilidad)

SHAP (SHapley Additive exPlanations) descompone la predicción de cada muestra en contribuciones individuales de cada feature, basándose en la teoría de juegos cooperativos:

$$\hat{y}_i = \phi_0 + \sum_{j=1}^d \phi_{ij}$$

Donde $\phi_0$ es el valor base (media del target) y $\phi_{ij}$ es la contribución de la feature $j$ a la predicción de la muestra $i$.

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Dataset tabular **mediano o grande** con relaciones no lineales.
- El target tiene **distribución asimétrica** (Tweedie, Poisson).
- Necesitas el **mejor RMSE posible** en tabular.
- Hay **valores faltantes** (manejo nativo).

### ❌ NO lo uses cuando:
- Necesitas **extrapolación** (como todo árbol, no extrapola).
- Dataset muy pequeño (< 500 muestras).
- Series temporales sin features de lag (usar modelos específicos).

## 5. Problemas Específicos del Regresor

### 5.1 Outliers en el Target
Con MSE, un outlier extremo domina el entrenamiento.
**Solución:** Usar `objective='reg:absoluteerror'` (XGB) o transformar el target con `log1p`.

### 5.2 Escalado del Target
XGBoost y LightGBM son **invariantes al escalado de features**, pero escalar el target puede mejorar la estabilidad numérica con `learning_rate` muy pequeño.

### 5.3 Sobreajuste sin Early Stopping
Con `n_estimators=1000` y `learning_rate=0.3` sin early stopping el modelo memorizará el training set.
**Regla:** Siempre usar early stopping con un conjunto de validación dedicado.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print(f'XGBoost v{xgb.__version__} | LightGBM v{lgb.__version__}')

---
## 6. Demostración en 1D — Cómo el Boosting Corrige Residuos

In [ ]:
# =============================================================================
# DATOS SINTÉTICOS 1D: visualizar la naturaleza iterativa del boosting
# Mostramos cómo cada árbol se ajusta a los residuos del modelo anterior
# =============================================================================
rng = np.random.RandomState(SEED)
X_1d = np.sort(rng.uniform(0, 6, 150)).reshape(-1, 1)
y_1d = np.sin(X_1d.ravel() * 1.5) * 2 + rng.randn(150) * 0.3
x_plot = np.linspace(0, 6, 400).reshape(-1, 1)

n_iter_show = [1, 5, 20, 100]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, n_iter in zip(axes.ravel(), n_iter_show):
    model = xgb.XGBRegressor(
        n_estimators=n_iter, learning_rate=0.3, max_depth=2,
        random_state=SEED, verbosity=0
    )
    model.fit(X_1d, y_1d)
    y_hat = model.predict(x_plot)
    r2 = r2_score(y_1d, model.predict(X_1d))

    ax.scatter(X_1d, y_1d, s=15, alpha=0.5, color='gray', label='Datos')
    ax.plot(x_plot, y_hat, color='steelblue', lw=2.5, label=f'XGB ({n_iter} árboles)')
    ax.plot(x_plot, np.sin(x_plot.ravel()*1.5)*2, 'k--', lw=1.2, alpha=0.5, label='Función real')
    ax.set_title(f'n_estimators={n_iter} | Train R²={r2:.3f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Evolución del Boosting: Más Árboles → Mejor Ajuste',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Regresión Real — California Housing

In [ ]:
# =============================================================================
# CARGA Y SPLITS
# =============================================================================
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)
print(f'Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
# =============================================================================
# XGBOOST REGRESOR con Early Stopping
# =============================================================================
xgb_reg = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.1,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=50,
    random_state=SEED,
    n_jobs=-1
)
xgb_reg.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

y_pred_xgb = xgb_reg.predict(X_test)
print(f'XGBoost Regresor:')
print(f'  Árboles usados : {xgb_reg.best_iteration + 1}')
print(f'  R²             : {r2_score(y_test, y_pred_xgb):.4f}')
print(f'  RMSE           : {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}')
print(f'  MAE            : {mean_absolute_error(y_test, y_pred_xgb):.4f}')

In [ ]:
# =============================================================================
# LIGHTGBM REGRESOR con Early Stopping
# =============================================================================
lgb_reg = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    feature_fraction=0.7,
    lambda_l2=2.0,
    lambda_l1=0.1,
    min_child_samples=20,
    objective='regression',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
lgb_reg.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)

y_pred_lgb = lgb_reg.predict(X_test)
print(f'LightGBM Regresor:')
print(f'  Árboles usados : {lgb_reg.best_iteration_}')
print(f'  R²             : {r2_score(y_test, y_pred_lgb):.4f}')
print(f'  RMSE           : {np.sqrt(mean_squared_error(y_test, y_pred_lgb)):.4f}')
print(f'  MAE            : {mean_absolute_error(y_test, y_pred_lgb):.4f}')

In [ ]:
# =============================================================================
# VISUALIZACIÓN 1: Curvas de Aprendizaje (train vs val RMSE)
# =============================================================================
xgb_val_rmse = xgb_reg.evals_result()['validation_0']['rmse']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(xgb_val_rmse, color='steelblue', lw=1.5, label='XGBoost — Val RMSE')
ax.axvline(xgb_reg.best_iteration, color='steelblue', linestyle='--', alpha=0.7,
           label=f'Early stop iter={xgb_reg.best_iteration+1}')
ax.set_xlabel('Número de árboles', fontsize=12)
ax.set_ylabel('RMSE (validación)', fontsize=12)
ax.set_title('Curva de Aprendizaje — XGBoost Regresor', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 2: Predichos vs Reales + Residuos
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for row, (name, y_pred) in enumerate([('XGBoost', y_pred_xgb), ('LightGBM', y_pred_lgb)]):
    residuals = y_test.values - y_pred
    r2 = r2_score(y_test, y_pred)

    # Scatter predichos vs reales
    axes[row, 0].scatter(y_test, y_pred, alpha=0.2, s=8, color='steelblue' if row==0 else 'seagreen')
    lim = [y_test.min(), y_test.max()]
    axes[row, 0].plot(lim, lim, 'r--', lw=2)
    axes[row, 0].set_title(f'{name} — Predichos vs Reales  ($R^2$={r2:.4f})',
                            fontsize=11, fontweight='bold')
    axes[row, 0].set_xlabel('Reales'); axes[row, 0].set_ylabel('Predicciones')

    # Residuos
    axes[row, 1].hist(residuals, bins=60, edgecolor='white',
                      color='steelblue' if row==0 else 'seagreen', alpha=0.7)
    axes[row, 1].axvline(0, color='crimson', lw=2, linestyle='--')
    axes[row, 1].set_title(f'{name} — Residuos  (media={residuals.mean():.4f})',
                            fontsize=11, fontweight='bold')
    axes[row, 1].set_xlabel('Residuo')

plt.suptitle('Evaluación Comparativa — California Housing',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 3: Importancia de Features — LightGBM
# LightGBM ofrece 'split' (frecuencia) y 'gain' (mejora promedio de pérdida)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, imp_type in zip(axes, ['split', 'gain']):
    imp = pd.Series(
        lgb_reg.booster_.feature_importance(importance_type=imp_type),
        index=housing.feature_names
    ).sort_values(ascending=False)
    ax.barh(imp.index[::-1], imp.values[::-1], color='seagreen', alpha=0.8)
    ax.set_title(f'LightGBM Importancia ({imp_type})', fontsize=12, fontweight='bold')
    ax.set_xlabel(imp_type)

plt.suptitle('Importancia de Features — LightGBM Regresor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SHAP VALUES — Explicabilidad de las predicciones individuales
# SHAP descompone cada predicción en contribuciones por feature
# =============================================================================
try:
    import shap
    explainer  = shap.TreeExplainer(xgb_reg)
    shap_vals  = explainer.shap_values(X_test)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Summary plot (beeswarm) — visión global
    plt.sca(axes[0])
    shap.summary_plot(shap_vals, X_test, plot_type='dot', show=False, max_display=10)
    axes[0].set_title('SHAP — Summary Plot (beeswarm)', fontsize=12, fontweight='bold')

    # Bar plot — importancia global por SHAP
    plt.sca(axes[1])
    shap.summary_plot(shap_vals, X_test, plot_type='bar', show=False, max_display=10)
    axes[1].set_title('SHAP — Importancia Media |SHAP|', fontsize=12, fontweight='bold')

    plt.suptitle('SHAP Values — XGBoost Regresor (California Housing)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('SHAP no instalado. Instalar con: pip install shap')
    print('SHAP es altamente recomendable para interpretar modelos GB en producción.')

In [ ]:
# =============================================================================
# COMPARATIVA FINAL: sklearn GBM vs XGBoost vs LightGBM
# =============================================================================
from sklearn.ensemble import GradientBoostingRegressor

models_reg = {
    'sklearn GBM':  GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                               max_depth=4, random_state=SEED),
    'XGBoost':      xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                                      max_depth=4, random_state=SEED, verbosity=0, n_jobs=-1),
    'LightGBM':     lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                       num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1)
}

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'{"Modelo":<16}  {"R² (CV)":>12}  {"RMSE (CV)":>12}')
print('-' * 45)
for name, model in models_reg.items():
    r2_cv   = cross_val_score(model, X, y, cv=kf, scoring='r2')
    rmse_cv = np.sqrt(-cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error'))
    print(f'{name:<16}  {r2_cv.mean():.4f}±{r2_cv.std():.3f}  '
          f'{rmse_cv.mean():.4f}±{rmse_cv.std():.3f}')

---
## 8. Resumen y Checklist

| Aspecto | XGBoost | LightGBM |
|---|---|---|
| **Pérdida default** | `reg:squarederror` (MSE) | `regression` (MSE) |
| **Parámetro clave** | `max_depth` | `num_leaves` |
| **Outliers en target** | `reg:absoluteerror` | `regression_l1` |
| **Velocidad en grandes datos** | Rápido | Más rápido |
| **SHAP nativo** | ✅ | ✅ |

### ✅ Checklist Regresor:
- [ ] ¿Usé early stopping con validación dedicada?
- [ ] ¿Hay outliers en el target? → cambiar `objective` a MAE/Huber
- [ ] ¿Grafiqué predichos vs reales y distribución de residuos?
- [ ] ¿Usé SHAP para interpretar predicciones individuales?
- [ ] ¿Comparé con un baseline lineal (Ridge)?

---
*Ver notebook de clasificación: `gb_clasificador.ipynb`*